<a href="https://colab.research.google.com/github/nguyenduyvu61107/.BTNHOMKIA/blob/main/nh%E1%BA%ADn_di%E1%BB%87n_food.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
# Lấy IP để lát điền vào localtunnel
!curl ipv4.icanhazip.com

# Chạy app
!streamlit run app.py & npx localtunnel --port 8501


35.229.137.140
/bin/bash: line 1: streamlit: command not found
⠙⠹⠸⠼⠴⠦Need to install the following packages:
localtunnel@2.0.2
Ok to proceed? (y) y

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇your url is: https://cool-beers-dream.loca.lt
^C


In [ ]:
%%writefile app.py
import streamlit as st
import pandas as pd
from PIL import Image
import tensorflow as tf
import numpy as np

# =========================
# CẤU HÌNH TRANG & CSS
# =========================
st.set_page_config(page_title="Food Recognition System", page_icon="🍱", layout="wide")
st.markdown("""
<style>
.main{ background-color:#F7F9FC; }
h1{ color:#1E3A8A; }
.stMetric{ background-color:white; padding:15px; border-radius:15px; }
div[data-testid="stSidebar"]{ background-color:#E8EEF9; }
</style>
""", unsafe_allow_html=True)

# ====================================================
# TẢI MODEL AI (CHẠY TRÊN COLAB)
# ====================================================
@st.cache_resource  # Dùng cái này để Streamlit chỉ load model 1 lần duy nhất, ko bị load lại khi bấm nút
def load_my_model():
    # Gọi đúng tên file .keras của b
    return tf.keras.models.load_model('canteen_cnn_11classes.keras')

try:
    model = load_my_model()
except Exception as e:
    st.error(f"Lỗi load model: {e}. Nhớ upload file .keras lên Colab nha b!")

# Định nghĩa 11 món ăn tương ứng với đầu ra của Model (BẬT MÍ: b nhớ sửa lại đúng thứ tự lớp và giá tiền của b nha)
FOOD_MENU = {
    0: {"name": "thit_kho", "price": 25000},
    1: {"name": "rau_xao", "price": 10000},
    2: {"name": "com_trang", "price": 10000},
    3: {"name": "ca_kho", "price": 25000},
    4: {"name": "canh_chua", "price": 12000},
    5: {"name": "trung_chien", "price": 10000},
    6: {"name": "ga_chiên", "price": 30000},
    7: {"name": "dau_hu_nhoi_thit", "price": 20000},
    8: {"name": "su_su_xao", "price": 10000},
    9: {"name": "thit_luoc", "price": 22000},
    10: {"name": "canh_bi_dao", "price": 8000}
}

# Hàm xử lý ảnh và Predict thật
def predict_food(image):
    # 1. Resize ảnh về đúng kích thước mà model CNN của b yêu cầu (ví dụ thường là 224x224 hoặc 150x150)
    # B xem lúc train b để size bao nhiêu thì sửa lại số ở đây nha (ví dụ: 224, 224)
    img_size = (128, 128)
    img = image.resize(img_size)

    # 2. Chuyển ảnh thành mảng numpy và chuẩn hóa (Nếu lúc train b chia 255.0)
    img_array = np.array(img) / 255.0

    # 3. Thêm dimension batch (Keras yêu cầu input dạng [1, height, width, channels])
    img_array = np.expand_dims(img_array, axis=0)

    # 4. Dự đoán
    predictions = model.predict(img_array)

    # Giả sử model nhận diện đa nhãn hoặc trả về xác suất của 11 lớp.
    # Tớ lấy ví dụ cách lấy những món có độ tin cậy > 0.5 (bạn có thể chỉnh lại tùy cấu trúc model):
    results = []

    # Trường hợp là Single-label (chỉ ra 1 món có xác suất cao nhất)
    # class_idx = np.argmax(predictions[0])
    # confidence = predictions[0][class_idx]
    # if confidence > 0.5:
    #      results.append({"name": FOOD_MENU[class_idx]["name"], "confidence": float(confidence), "price": FOOD_MENU[class_idx]["price"]})

    # Trường hợp Multi-label hoặc b muốn test thử lấy các món có score cao:
    for idx, score in enumerate(predictions[0]):
        if score > 0.4:  # Ngưỡng tin cậy > 40% thì lấy
            results.append({
                "name": FOOD_MENU[idx]["name"],
                "confidence": round(float(score), 2),
                "price": FOOD_MENU[idx]["price"]
            })

    # Phòng hờ nếu ko ra món nào thì cho món mặc định
    if not results:
        results.append({"name": "Chưa nhận diện được", "confidence": 0.0, "price": 0})

    return results

# =========================
# GIAO DIỆN CHÍNH (SIDEBAR & MAIN)
# =========================
st.sidebar.title("⚙️ Input")
input_option = st.sidebar.radio("Chọn phương thức", ["📁 Upload ảnh", "📷 Camera"])

image = None
if input_option == "📁 Upload ảnh":
    uploaded_file = st.file_uploader("Tải ảnh khay cơm", type=["jpg","jpeg","png"])
    if uploaded_file:
        image = Image.open(uploaded_file).convert("RGB") # Convert RGB tránh lỗi ảnh PNG trong suốt
else:
    st.sidebar.warning("⚠️ Chạy trên Colab camera có thể không hoạt động ổn định.")
    camera_image = st.camera_input("Chụp ảnh khay cơm")
    if camera_image:
        image = Image.open(camera_image).convert("RGB")

st.title("🍱 Food Recognition System")

if image is not None:
    results = predict_food(image)
    total_price = sum(item["price"] for item in results)

    col1, col2 = st.columns([1.2, 1])

    with col1:
        st.subheader("Ảnh khay cơm")
        st.image(image, use_container_width=True)

    with col2:
        st.subheader("Danh sách món")
        df = pd.DataFrame(results)
        df.columns = ["Tên món", "Độ tin cậy", "Giá tiền"]
        st.dataframe(df, use_container_width=True)
        st.metric("💰 Tổng tiền", f"{total_price:,} VND")

        # QR THANH TOÁN
        BANK_ID = "VCB"
        ACCOUNT_NUMBER = "123456789"
        ACCOUNT_NAME = "NGUYEN VAN A"
        qr_url = f"https://img.vietqr.io/image/{BANK_ID}-{ACCOUNT_NUMBER}-compact2.png?amount={total_price}&addInfo=KHAYCOM&accountName={ACCOUNT_NAME}"
        st.subheader("Thanh toán QR")
        st.image(qr_url, width=300)

    st.markdown("---")
    invoice_text = ""
    for i, item in enumerate(results, start=1):
        invoice_text += f"{i}. {item['name']} - {item['price']:,} VND\n"
    invoice_text += f"\nTổng tiền: {total_price:,} VND"

    st.download_button("📄 Xuất hóa đơn", invoice_text, file_name="bill.txt")

if st.button("🔄 Reset"):
    st.session_state.clear()
    st.rerun()

In [6]:
# Lấy IP để lát điền vào localtunnel
!curl ipv4.icanhazip.com

# Chạy app
!streamlit run app.py & npx localtunnel --port 8501

35.229.137.140
/bin/bash: line 1: streamlit: command not found
⠙⠹⠸⠼⠴⠦⠧your url is: https://tricky-zoos-nail.loca.lt
^C


In [8]:
!pip install pyngrok -q
from pyngrok import ngrok



In [12]:
!streamlit run app.py --server.port 8501 &



2026-06-21 01:51:11.889 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://35.229.137.140:8501



  Stopping...


In [ ]:
# 1. Điền token của b vào đây để xác thực
NGROK_TOKEN = "3FQThF9rElmdcAJCU71C2xmETkj_2FJ1wycu4YuuVqqrqM6W2"
ngrok.set_auth_token(NGROK_TOKEN)

# 2. Tạo kết nối tunnel đến cổng 8501 của Streamlit
pub_url = ngrok.connect(8501)
print("Link vào app của b đây nè:", pub_url.public_url)

# 3. Chạy Streamlit
!streamlit run app.py

Link vào app của b đây nè: https://drop-down-batboy-passing.ngrok-free.dev




2026-06-21 01:58:11.657 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://35.229.137.140:8501

2026-06-21 01:58:18.611059: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-06-21 01:58:18.838626: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-06-21 01:58:25.834115: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 536ms/step
2026-06-21 01:58:37.792 Please replace `use_container_width` with `width`.

`u

In [11]:
!pip install streamlit pandas pillow pyngrok -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 24.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 57.9 MB/s eta 0:00:00
